In [ ]:
import sys
sys.path.append("../")
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from odometry.datasets.map_handler import MapHandler
from odometry.datasets.radnav_ds import radnavDS
from odometry.plotting.plotter_localization import PlotterLocalization
from odometry.point_cloud_processing.vel_filtering import VelFiltering

In [ ]:
from dotenv import load_dotenv
import os

#loading enviroment variables
load_dotenv()
DATASET_PATH = os.getenv("DATASET_DIRECTORY")
MAP_DIRECTORY = os.getenv("MAP_DIRECTORY")

dataset = radnavDS(
    dataset_path=DATASET_PATH + "/WILK_Long_Movement",
    radar_folder="radar_combined",
    lidar_folder="lidar",
    camera_folder="camera",
    imu_orientation_folder="imu_data",
    imu_full_folder="imu_data_full",
    vehicle_vel_folder="vehicle_vel"
)

map_handler = MapHandler(
    maps_folder=MAP_DIRECTORY,
    map_file="wilkinson.yaml"
)

plotter = PlotterLocalization(dataset,map_handler)

In [ ]:
idx = 200
plt.imshow(dataset.get_camera_frame(idx))

In [ ]:
idx = 200

#get the point cloud
ego_x_vel = dataset.get_vehicle_vel_data(idx)[0,1]
vehicle_vel = np.array([ego_x_vel,0.0])
print(ego_x_vel)

#get the detections
detections = dataset.get_radar_detections(idx)
print(detections.shape)

#plot the detections
plotter.plot_detections(detections, show=True)

In [ ]:
vel_filter = VelFiltering(v_thresh=.4)

#plot dynamic detections
dynamic_objects = vel_filter.get_dynamic_detections(detections,vehicle_vel)
plotter.plot_detections(dynamic_objects, show=True)

In [ ]:
#plot the static objects
static_objects = vel_filter.get_static_detections(detections,vehicle_vel)
plotter.plot_detections(static_objects,show=True)

# Dataset evaluation

In [ ]:
from sklearn.cluster import DBSCAN

#new code for clustering
dbscan_clusterer:DBSCAN = DBSCAN(
    eps=1.0,
    min_samples=7
)
min_distance_to_cluster = 2.0
#end new code for clustering

vel_filter = VelFiltering(v_thresh=.1)
num_rejections = []
num_new_rejections = []
num_original = []
percent = []
clusters = []

for i in tqdm(range(dataset.num_frames)):

    ego_x_vel = dataset.get_vehicle_vel_data(i)[0,1]
    vehicle_vel = np.array([ego_x_vel,0.0])

    detections = dataset.get_radar_detections(i)
    original_num_dets = detections.shape[0]
    num_original.append(original_num_dets)

    static_objects = vel_filter.get_static_detections(detections,vehicle_vel)

    #start of new code
    dynamic_objects = vel_filter.get_dynamic_detections(detections,vehicle_vel)

    if dynamic_objects.shape[0] > 0:
        labels = dbscan_clusterer.fit_predict(dynamic_objects[:,0:2])
        
        #get the unique cluster ids
        unique_labels = np.unique(labels)
        num_clusters = len(unique_labels) -1
        clusters.append(num_clusters)

        #create a list of valid indicies
        valid_idxs = np.ones(static_objects.shape[0],dtype=bool)

        if num_clusters > 0:
            for cluster_id in unique_labels[1:]:
                
                #identify points in the cluster
                cluster_points = dynamic_objects[labels==cluster_id,0:2]

                #compute the centroid
                centroid = np.average(cluster_points,axis=0)

                #compute distance between centroid and all static objects
                distances = np.linalg.norm(static_objects[:,0:2] - centroid)

                moving_idxs = distances < min_distance_to_cluster

                valid_idxs[moving_idxs] = False

            static_objects = static_objects[valid_idxs]

            num_new_rejections.append(valid_idxs.shape[0] - np.sum(valid_idxs))
        else:
            num_new_rejections.append(0)


    else:
        clusters.append(0)
        num_new_rejections.append(0)
    #end of new code

    num_static_objs = static_objects.shape[0]
    rejections = original_num_dets - num_static_objs
    num_rejections.append(rejections)

    percent_rejected = float(rejections/original_num_dets)
    percent.append(percent_rejected)

# plt.plot(num_rejections)
plt.plot(num_rejections)
plt.plot(num_new_rejections)
plt.show()


### Debugging

Useful for looking at what points are being rejected

In [ ]:
#predict the velocity measurement
v_pred = vel_filter.predict_vels(detections[:,0:2],vehicle_vel)
print(v_pred)

In [ ]:
#print the true values
v_true = detections[:,3]
print(detections.shape)
print(v_true)

In [ ]:

errors = vel_filter.square_error_loss(v_true,v_pred)
print(errors)